In [12]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    BooleanType,
    DoubleType,
    StringType,
    StructField,
    StructType,
)

# Initialize Spark session (Colab-compatible)
spark = SparkSession.builder.appName("MetadataControlTable").getOrCreate()

# ---------------------------------------
# STEP 1: Drop table if exists
# ---------------------------------------
spark.sql("DROP TABLE IF EXISTS control_table")

# ---------------------------------------
# STEP 2: Create config DataFrame schema
# ---------------------------------------
schema = StructType(
    [
        StructField("source_name", StringType(), True),
        StructField("entity_name", StringType(), True),
        StructField("source_type", StringType(), True),
        StructField("source_path", StringType(), True),
        StructField("load_type", StringType(), True),
        StructField("watermark_column", StringType(), True),
        StructField("last_processed_value", StringType(), True),
        StructField("mapping_config_path", StringType(), True),
        StructField("transformation_module", StringType(), True),
        StructField("schedule_type", StringType(), True),
        StructField("active_flag", BooleanType(), True),
        StructField("config_version", StringType(), True),
        StructField("data_quality_threshold", DoubleType(), True),
    ]
)

# ---------------------------------------
# STEP 3: Insert sample records
# ---------------------------------------
records = [
    ("carrier_crm", "sales_transactions", "csv", "/data/carrier_crm/", "full", None, None, "/config/carrier_crm_sales.json", None, "daily", True, "v1", 0.9),
    ("carrier_crm", "customers", "csv", "/data/carrier_crm/", "full", None, None, "/config/carrier_crm_customers.json", None, "daily", True, "v1", 0.9),
    ("carrier_crm", "policies", "csv", "/data/carrier_crm/", "full", None, None, "/config/carrier_crm_policies.json", None, "daily", True, "v1", 0.9),
    ("oas_db", "sales_transactions", "db", "jdbc://oas_db", "incremental", "updated_at", None, "/config/oas_sales.json", None, "hourly", True, "v1", 0.9),
    ("oas_db", "customers", "db", "jdbc://oas_db", "incremental", "updated_at", None, "/config/oas_customers.json", None, "hourly", True, "v1", 0.9),
    ("oas_db", "policies", "db", "jdbc://oas_db", "incremental", "updated_at", None, "/config/oas_policies.json", None, "hourly", True, "v1", 0.9),
]

# ---------------------------------------
# STEP 4: Create DataFrame
# ---------------------------------------
control_df = spark.createDataFrame(records, schema=schema)

# ---------------------------------------
# STEP 5: Register table
# ---------------------------------------
control_df.createOrReplaceTempView("control_table")

# OUTPUT: show DataFrame and print final table
print("Control table DataFrame:")
control_df.show(truncate=False)

print("Final table from Spark SQL:")
spark.sql("SELECT * FROM control_table").show(truncate=False)



Control table DataFrame:
+-----------+------------------+-----------+------------------+-----------+----------------+--------------------+----------------------------------+---------------------+-------------+-----------+--------------+----------------------+
|source_name|entity_name       |source_type|source_path       |load_type  |watermark_column|last_processed_value|mapping_config_path               |transformation_module|schedule_type|active_flag|config_version|data_quality_threshold|
+-----------+------------------+-----------+------------------+-----------+----------------+--------------------+----------------------------------+---------------------+-------------+-----------+--------------+----------------------+
|carrier_crm|sales_transactions|csv        |/data/carrier_crm/|full       |NULL            |NULL                |/config/carrier_crm_sales.json    |NULL                 |daily        |true       |v1            |0.9                   |
|carrier_crm|customers         |csv

In [10]:
from pyspark.sql.functions import when, col

df = spark.table("control_table")

df = df.withColumn(
    "source_path",
    when(col("source_name") == "carrier_crm", "/content/carrier_crm_source.csv")
    .otherwise(col("source_path"))
)

df = df.withColumn(
    "mapping_config_path",
    when((col("source_name") == "carrier_crm") & (col("entity_name") == "customers"),
         "/content/carrier_crm_customers.json")
    .when((col("source_name") == "carrier_crm") & (col("entity_name") == "sales_transactions"),
         "/content/carrier_crm_sales.json")
    .when((col("source_name") == "carrier_crm") & (col("entity_name") == "policies"),
         "/content/carrier_crm_policies.json")
    .otherwise(col("mapping_config_path"))
)

df.createOrReplaceTempView("control_table")

In [4]:
"""Utilities for loading source runtime configuration from Delta control tables."""

from __future__ import annotations

import json
import logging
from typing import Any, Dict

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F

LOGGER = logging.getLogger(__name__)


class ConfigNotFoundError(LookupError):
    """Raised when no active configuration exists for a source."""


class ConfigLoader:
    """Load active source configuration records from a Delta control table.

    Parameters
    ----------
    spark:
        Active :class:`SparkSession` used to query the control table.
    table_name:
        Fully-qualified Delta table name that stores source configurations.
    """

    def __init__(self, spark: SparkSession, table_name: str = "config.control_table") -> None:
        self.spark = spark
        self.table_name = table_name

    def load_config(self, source_name: str) -> Dict[str, Any]:
        """Fetch an active source configuration and return it as a dictionary.

        Parameters
        ----------
        source_name:
            Unique name of the data source whose active config is requested.

        Returns
        -------
        dict[str, Any]
            Source configuration represented as a native Python dictionary.

        Raises
        ------
        ValueError
            If ``source_name`` is blank.
        ConfigNotFoundError
            If no active configuration record exists for ``source_name``.
        """

        normalized_source_name = source_name.strip() if source_name else ""
        if not normalized_source_name:
            raise ValueError("source_name must be a non-empty string.")

        config_df = self._build_active_config_query(normalized_source_name)
        config_row = config_df.limit(1).collect()
        if not config_row:
            raise ConfigNotFoundError(
                f"No active configuration found in {self.table_name} for source_name='{normalized_source_name}'."
            )

        config = config_row[0].asDict(recursive=True)
        LOGGER.info(
            "Loaded active config for source_name='%s' from %s: %s",
            normalized_source_name,
            self.table_name,
            json.dumps(config, sort_keys=True, default=str),
        )
        return config

    def _build_active_config_query(self, source_name: str) -> DataFrame:
        """Construct the active configuration query DataFrame for a source."""

        return (
            self.spark.table(self.table_name)
            .filter((F.col("source_name") == source_name) & (F.col("active_flag") == F.lit(True)))
        )


In [8]:
config = ConfigLoader(spark, "config.control_table")
carrier_config_dic = config.load_config("carrier_crm")

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `config`.`control_table` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01;
'UnresolvedRelation [config, control_table], [], false
